In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install mediapipe

In [ ]:
from google.colab.patches import cv2_imshow
from google.colab import files

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(min_detection_confidence=0.5, min_tracking_confidence=0.5)
cap = cv2.VideoCapture('/content/drive/MyDrive/연구과제/스마트침구연구과제/t5.mp4')

In [ ]:
width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
fps = cap.get(cv2.CAP_PROP_FPS)

In [ ]:
print('가로: ', str(width))
print('세로: ', str(height))
print('총 프레임수: ', str(count))
print('FPS: ', str(fps))

In [ ]:
num = 0

while cap.isOpened():
    success, image = cap.read()
    if not success:
        break
    image = cv2.cvtColor(cv2.flip(image, 1), cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = face_mesh.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    # multi_face_landmarks : 각 면이 468개의 얼굴 랜드마크 리스트로 표현
    # landmark 에서 원하는 랜드마크 인덱스를 통해 좌표를 가져오고 x,y,z를 통해 값을 가져옴
    # x와 y는 정규화된 값으로 0~1 사이의 값을 가진다. 
    # z는 Mesh 중앙을 지나는 평면을 기준으로 하는 상대적인 깊이를 나타냄.
    if results.multi_face_landmarks:
        for face_landmarks in results.multi_face_landmarks:
            for landmark in face_landmarks.landmark:
                x1 = int(landmark.x * image.shape[1])
                y1 = int(landmark.y * image.shape[0])
                # x, y 좌표 화면에 그리기
                cv2.circle(image, (x1, y1), 2, (0, 255, 0), -1)

    img_h, img_w, img_c = image.shape
    face_3d = []
    face_2d = []

    if results.multi_face_landmarks:
        edge_indices = [10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288, 397, 365, 379, 378, 400, 377, 152, 148, 176, 149, 150, 136, 172, 58, 132, 93, 234, 127, 162, 21, 54, 103, 67, 109, 10]
        face_landmark_list = [33, 263, 1, 61, 291, 199]     # 얼굴 특징점 랜드마트 인덱스 리스트
        
        edge_2d = []    
        edge_3d = []

        focal_length = 1 * img_w
        cam_matrix = np.array([ [focal_length, 0, img_h / 2],
                                [0, focal_length, img_w / 2],
                                [0, 0, 1]])
        dist_matrix = np.zeros((4, 1), dtype=np.float64)


        for face_landmarks in results.multi_face_landmarks:
            for idx, lm in enumerate(face_landmarks.landmark):
                if idx in face_landmark_list: 
                    if idx == 1:
                        nose_2d = (lm.x * img_w, lm.y * img_h)     #이미지 상의 코의 좌표
                        # print("Nose Position (lm)=> ", lm.x, lm.y)
                        # print("Nose Position (Pr)=> ", lm.x * img_w, lm.y * img_h)
                        nose_x = int(lm.x * 20)
                        nose_y = int(lm.y * 20)
                        print("Nose Position (Pr)=> ", nose_x, nose_y)
                        # nose_3d = (lm.x * img_w, lm.y * img_h, lm.z * 8000)
                        nose_3d = (lm.x * 20, lm.y * 20, lm.z * 200000)
                        # print("nose_3d:", nose_3d)

                        # 20x20 배열 생성
                        init_arr = np.zeros((20,20))
                        df = pd.DataFrame(init_arr)

                        # (코) 20x20 배열 원소 값 채워주기 -> 코의 중심점으로부터의 거리 계산
                        for i in range (0,20) :
                          for j in range (0,20) :
                            df.loc[i,j] = math.sqrt((i-nose_x)**2 + (j-nose_y)**2)
                        # print(df)
                        
                        df = df.T

                        # (코 중심) heatmap 그려주기
                        plt.matshow(df)
                        plt.xticks(np.arange(0.5, len(df.columns), 1), df.columns)
                        plt.yticks(np.arange(0.5, len(df.index), 1), df.index)
                        plt.title('Nose Heatmap', fontsize=20)
                        plt.colorbar()

                        # (움짤) 히트맵 저장
                        plt.savefig(f"heatmap_{num}.png")

                        # import matplotlib.image as mpimg
                        # img = mpimg.imread(f"heatmap_{num}.png")

                        plt.show()
                        
                    x, y = int(lm.x * img_w), int(lm.y * img_h)
                    face_2d.append([x, y])
                    face_3d.append([x, y, lm.z])

            face_2d = np.array(face_2d, dtype=np.float64)
            face_3d = np.array(face_3d, dtype=np.float64) 


             # (추가한 부분) 테두리 라인 따기
            for idx in edge_indices:
                landmark = face_landmarks.landmark[idx]

                x = int(landmark.x * img_w)
                y = int(landmark.y * img_h)
                edge_2d.append([x, y])               # 테두리 라인 2d 좌표
                edge_3d.append([x, y, landmark.z])

            # top_point와 bottom_point 구하기
            top = edge_2d[0]
            bottom = edge_2d[18]
            #print(top, bottom)   #얼굴 세로 길이의 양 끝 점
            cv2.circle(image, (top[0], top[1]), 5, (255, 255, 255), -1)
            cv2.circle(image, (bottom[0], bottom[1]), 5, (255, 255, 255), -1)


            migan_x = int(top[0] + abs(bottom[0]-top[0])*0.3)
            migan_y = int(top[1] + abs(bottom[1]-top[1])*0.3)
            cv2.circle(image, (migan_x, migan_y), 5, (255, 255, 255), -1)
            # print(int(top[0] + abs(bottom[0]-top[0])*0.3), int(top[1] + abs(bottom[1]-top[1])*0.3))  #미간의 실제 이미지 상의 좌표

            mg_x = migan_x/(width/20)
            mg_y = migan_y/(height/20)
            # print("미간 좌표 ->", mg_x, mg_y)

            # # 20x20 미간 배열 생성
            # migan_arr = np.zeros((20,20))
            # m_df = pd.DataFrame(migan_arr)

            # # (미간) 20x20 배열 원소 값 채워주기 -> 미간의 중심점으로부터의 거리 계산
            # for i in range (0,20) :
            #   for j in range (0,20) :
            #     #print(i, j, abs(i-nose_x) + abs(j-nose_y))
            #     # df.loc[i,j] = abs(i-nose_x) + abs(j-nose_y)
            #     m_df.loc[i,j] = math.sqrt((i-mg_x)**2 + (j-mg_y)**2)
            # # print(m_df)

            # m_df = m_df.T

            # # 미간 중심 heatmap 그려주기
            # plt.matshow(m_df)
            # plt.xticks(np.arange(0.5, len(m_df.columns), 1), m_df.columns)
            # plt.yticks(np.arange(0.5, len(m_df.index), 1), m_df.index)
            # plt.title('Migan Heatmap plt.matshow()', fontsize=20)
            # plt.colorbar()
            # plt.show()

                
        # (추가한 부분) 2d 테두리 라인 그리기
        edge_2d = np.array(edge_2d)     
        cv2.polylines(image, [edge_2d], True, (0, 255, 0), 2)
        cv2.line(image, (top[0], top[1]), (bottom[0], bottom[1]), (0, 0, 255), 2)

        success, nose_rot_vec, nose_trans_vec = cv2.solvePnP(face_3d, face_2d, cam_matrix, dist_matrix)
        nose_3d_projection, jacobian = cv2.projectPoints(nose_3d, nose_rot_vec, nose_trans_vec, cam_matrix, dist_matrix)
        p1 = (int(nose_2d[0]), int(nose_2d[1]))
        p2 = (int(nose_3d_projection[0][0][0]), int(nose_3d_projection[0][0][1]))
        cv2.line(image, p1, p2, (255, 0, 0), 2)

        # 드라이브에 결과 이미지를 저장하는 코드
        path = '/content/drive/MyDrive/result_img/img_' + str(num) + '.png'
        cv2.imwrite(path, image)

        # num : 이미지, 히트맵 저장 숫자 관련 변수
        num += 1

    cv2_imshow(image)

    if cv2.waitKey(1) & 0xFF == 27:
        break
cap.release()

In [ ]:
# 드라이브에 저장한 이미지들을 움짤로 만드는 코드 (경로는 다 바꿔주어야 함)
# 결과물은 구글 드라이브에서 확인
from PIL import Image
import imageio

imgs = []
for i in range(0, num):
    imgs.append(Image.open(f'/content/drive/MyDrive/result_img/img_{i}.png'))

imageio.mimsave('/content/drive/MyDrive/result_video.gif', imgs, fps = 30.0)

# 히트맵 움짤 만드는 코드
h_imgs = []
for i in range(0, num):
    h_imgs.append(Image.open(f'/content/heatmap_{i}.png'))
imageio.mimsave('/content/drive/MyDrive/result_heatmap.gif', h_imgs, fps = 30.0)

In [ ]:
num

In [ ]:
# plt.matshow(m_df, marginal_x="histogram", marginal_y="histogram")

# import plotly.express as px

# # 그래프 그리기
# fig = px.density_heatmap(df, marginal_x="histogram", marginal_y="histogram", nbinsx=20, nbinsy=20)

# fig.show()